# Chunking: Facts and objectives 

Goal of this notebook:
- load the reconstructed saved HotpotQA corpus (See previous NoteBook 02)
- analyze document token lengths using the selected tokenizer
- finalaze chunk sizes of 32, 128, and 256 tokens based on the observed corpus distribution
- split documents into non overlapping token-based chunks 
- saved the resulting chunked corpora for later embedding, indexing, and retrieval experiments

Notes:
- these chunks will later be embedded and indexed in FAISS

In [23]:
# Load Corpus 
import json
from pathlib import Path

data_dir = Path("/home/a/arfaoui/rag_project/data")
corpus_path = data_dir / "hotpotqa_corpus_500.json"

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)
# Print to check 
print("Loaded corpus size:", len(corpus))
print("Example document:")
print(corpus[0])

Loaded corpus size: 4962
Example document:
{'title': 'Wedding Dress (film)', 'text': 'Wedding Dress is a South Korean drama film, released on January 14, 2010.'}


In [24]:
# We load a tokenizer to measure chunk size in TOKENS, not characters.
# Using a tokenizer makes chunking consistent and reproducible.


from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# print to check
print("Tokenizer loaded successfully.")
print("Tokenizer name:", MODEL_NAME)

Tokenizer loaded successfully.
Tokenizer name: meta-llama/Llama-3.2-1B-Instruct


In [25]:
# Quick sanity check on the tokenizer
test_text = corpus[0]["text"]
tokens = tokenizer.encode(test_text)

print(f"Test document title: {corpus[0]['title']}")
print(f"Test document text: {test_text}")
print(f"Character count: {len(test_text)}")
print(f"Token count: {len(tokens)}")
print(f"Ratio chars/tokens: {len(test_text)/len(tokens):.2f}")

Test document title: Wedding Dress (film)
Test document text: Wedding Dress is a South Korean drama film, released on January 14, 2010.
Character count: 73
Token count: 21
Ratio chars/tokens: 3.48


In [26]:
# Check token length distribution of corpus documents
# Before blindly applying chunk sizes from the first draft discussed with prof Luckow (128, 256),
# I first inspect the actual token length distribution of my corpus.
# This is important because:
# - Chunk sizes only matter if they actually split documents.
# - A chunk_size larger than most documents means no real chunking happens.
# - Choosing chunk sizes without this check leads to meaningless comparisons.
# - This analysis lets us decide chunk sizes RATIONALLY based on real data.
import numpy as np

token_lengths = [
    len(tokenizer.encode(doc["text"])) 
    for doc in corpus
]

print(f"Min tokens:    {np.min(token_lengths)}")
print(f"Max tokens:    {np.max(token_lengths)}")
print(f"Mean tokens:   {np.mean(token_lengths):.1f}")
print(f"Median tokens: {np.median(token_lengths):.1f}")
print(f"75th pct:      {np.percentile(token_lengths, 75):.1f}")
print(f"90th pct:      {np.percentile(token_lengths, 90):.1f}")
print(f"95th pct:      {np.percentile(token_lengths, 95):.1f}")

# How many docs would actually get split?
for size in [32, 64, 128, 256]:
    n_split = sum(1 for l in token_lengths if l > size)
    print(f"chunk_size={size} → {n_split}/{len(corpus)} docs would be split ({100*n_split/len(corpus):.1f}%)")

Min tokens:    12
Max tokens:    1397
Mean tokens:   128.6
Median tokens: 115.0
75th pct:      161.0
90th pct:      216.0
95th pct:      259.0
chunk_size=32 → 4858/4962 docs would be split (97.9%)
chunk_size=64 → 4186/4962 docs would be split (84.4%)
chunk_size=128 → 2041/4962 docs would be split (41.1%)
chunk_size=256 → 263/4962 docs would be split (5.3%)


### Observations  Token Length Distribution & Chunk Size Selection
chunk_size=256 splits only 5.3% of documents: almost no real chunking occurs,
making it a poor experimental choice on its own but useful as an upper bound
to observe the effect of very large context windows.

chunk_size=128 splits 41.1% of documents: a meaningful middle ground that
represents the standard default in RAG literature and covers the median document length.

chunk_size=32 splits 97.9% of documents: aggressive fine-grained chunking,
forces nearly every document to be split into multiple small pieces.

### Final chunk size selection: 32, 128, 256

- 32: fine grained, high splitting rate, precise but potentially loses context
- 128: balanced, aligns with corpus median, standard RAG reference point
- 256: coarse, minimal splitting, large context per chunk
### Why chunk_size=128 is used for the top-k analysis
When studying the effect of top-k (Analysis 1), we need to fix chunk size
to isolate the variable. chunk_size=128 is chosen because:
- It is one of the most commonly used default in RAG research
- It produces real chunking behavior on our corpus (41.1% of docs split)
- It sits at the corpus median -> neither too aggressive nor too coarse
- Results at this size are most comparable to existing RAG literature


In [27]:
# Token-based chunking function
# This splits one document into chunks based on TOKEN count, not character count.
# Overlap is fixed to 0 to avoid introducing another experimental factor.

def chunk_document(doc, doc_id, chunk_size, tokenizer, overlap=0):   # Overlap = 0 to avoid introducing another experimental factor.
    """
    Split one document into token-based chunks.

    Parameters
    ----------
    doc : dict
        A corpus document with keys like {"title": ..., "text": ...}
    doc_id : int
        Integer ID of the document in the corpus
    chunk_size : int
        Maximum number of tokens per chunk
    tokenizer : Hugging Face tokenizer
        Tokenizer used for token-based splitting
    overlap : int, default=0
        Number of overlapping tokens between consecutive chunks.
        Fixed to 0 here for experimental control.

    Returns
    -------
    chunks : list of dict
        List of chunk records with metadata
    """
    
    text = doc["text"]
    title = doc["title"]

    # Convert full document to token ids
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    chunk_index = 0

    # Safety check
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    while start < len(token_ids):
        end = start + chunk_size
        chunk_token_ids = token_ids[start:end]

        # Convert chunk token ids back to text
        chunk_text = tokenizer.decode(chunk_token_ids, skip_special_tokens=True).strip()

        if chunk_text:
            chunk_record = {
                "chunk_id": f"doc{doc_id}_chunk{chunk_index}",
                "doc_id": doc_id,
                "title": title,
                "chunk_index": chunk_index,
                "chunk_size": chunk_size,
                "overlap": overlap,
                "text": chunk_text,
                "n_tokens": len(chunk_token_ids)
            }
            chunks.append(chunk_record)

        start += chunk_size # No overlap, next chunk starts directly where the previous chunk ended 
        chunk_index += 1

    return chunks

In [10]:
# Test chunking on one example document 

test_doc = corpus[0]

test_chunks = chunk_document(
    doc=test_doc,
    doc_id=0,
    chunk_size=128,
    tokenizer=tokenizer,
    overlap=0
)

print("Document title:", test_doc["title"])
print("Original text length (chars):", len(test_doc["text"]))
print("Number of produced chunks:", len(test_chunks))

print("\nFirst chunk example:\n")
print(test_chunks[0])

Document title: Wedding Dress (film)
Original text length (chars): 73
Number of produced chunks: 1

First chunk example:

{'chunk_id': 'doc0_chunk0', 'doc_id': 0, 'title': 'Wedding Dress (film)', 'chunk_index': 0, 'chunk_size': 128, 'overlap': 0, 'text': 'Wedding Dress is a South Korean drama film, released on January 14, 2010.', 'n_tokens': 20}


In [28]:
# Inspect the first few chunks

for i, ch in enumerate(test_chunks[:3]):
    print(f"Chunk {i}")
    print("chunk_id:", ch["chunk_id"])
    print("n_tokens:", ch["n_tokens"])
    print("text preview:", ch["text"][:300])

Chunk 0
chunk_id: doc0_chunk0
n_tokens: 20
text preview: Wedding Dress is a South Korean drama film, released on January 14, 2010.


In [30]:
# Final chunking settings
# Overlap is fixed to 0 for all configurations.

chunk_sizes = [32, 128, 256]
print("Chunk sizes:", chunk_sizes)
# Apply token-based chunking  function (chunk_document) to the full corpus for each chunk size

all_chunked_corpora = {}

for chunk_size in chunk_sizes:
    print(f"\nWait for Chunk_size={chunk_size}")
    
    chunked_docs = []
    
    for doc_id, doc in enumerate(corpus):
        doc_chunks = chunk_document(
            doc=doc,
            doc_id=doc_id,
            chunk_size=chunk_size,
            tokenizer=tokenizer,
            overlap=0
        )
        chunked_docs.extend(doc_chunks)
    
    all_chunked_corpora[chunk_size] = chunked_docs
    print(f"Finished and total chunks created: {len(chunked_docs)}")

Chunk sizes: [32, 128, 256]

Wait for Chunk_size=32
Finished and total chunks created: 22186

Wait for Chunk_size=128
Finished and total chunks created: 7295

Wait for Chunk_size=256
Finished and total chunks created: 5236


In [31]:
# Inspect chunk statistics for each chunk size can be usefull later for evaluation or report
# Part of the learning process
# Not important 

for chunk_size, chunks in all_chunked_corpora.items():
    token_counts = [c["n_tokens"] for c in chunks]
    
    print(f"\n===== chunk_size = {chunk_size} =====")
    print(f"Total chunks: {len(chunks)}")
    print(f"Min tokens per chunk: {min(token_counts)}")
    print(f"Max tokens per chunk: {max(token_counts)}")
    print(f"Mean tokens per chunk: {np.mean(token_counts):.1f}")
    print(f"Median tokens per chunk: {np.median(token_counts):.1f}")


===== chunk_size = 32 =====
Total chunks: 22186
Min tokens per chunk: 1
Max tokens per chunk: 32
Mean tokens per chunk: 28.5
Median tokens per chunk: 32.0

===== chunk_size = 128 =====
Total chunks: 7295
Min tokens per chunk: 1
Max tokens per chunk: 128
Mean tokens per chunk: 86.8
Median tokens per chunk: 95.0

===== chunk_size = 256 =====
Total chunks: 5236
Min tokens per chunk: 1
Max tokens per chunk: 256
Mean tokens per chunk: 120.9
Median tokens per chunk: 111.0


## Observations from chunking

- Chunk size 32 produced the largest number of chunks 22,186, creating a much finer-grained retrieval space.
- Chunk size 128 produced 7,295 chunks, giving a more balanced index size and chunk granularity.
- Chunk size 256 produced 5,236 chunks, which is much closer to document-level retrieval because many original documents are shorter than 256 tokens.
- The mean chunk length decreases as chunk size increases in relative terms, because larger chunk sizes lead to more partially filled final chunks.
- The difference in total number of chunks across configurations is experimentally important: smaller chunk sizes increase the number of retrievable units, which may improve retrieval precision but also enlarge the search space.
- Since overlap was fixed to 0 for all settings, any later performance differences can be interpreted more cleanly as effects of chunk size rather than chunk redundancy.

In [32]:
# Save each chunked corpus to disk, this is very important so we can load the
# preprocessed chunks directly for the experiments.
# This avoids recomputing the chunks everytime.


for chunk_size, chunks in all_chunked_corpora.items():
    output_path = Path("/home/a/arfaoui/rag_project/data") / f"hotpotqa_chunks_{chunk_size}.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    
    print(f"Saved: {output_path}")

Saved: /home/a/arfaoui/rag_project/data/hotpotqa_chunks_32.json
Saved: /home/a/arfaoui/rag_project/data/hotpotqa_chunks_128.json
Saved: /home/a/arfaoui/rag_project/data/hotpotqa_chunks_256.json


## Chunking summary


Why this matters:
- chunked texts are the actual units that will be embedded and stored in FAISS
- each chunk will later be embedded and stored in a FAISS index
- later retrieval with top-k will operate on these chunks
- comparing chunk sizes helps analyze the tradeoff between retrieval granularity and contextual completeness
